# ExplainPlan Vision
## Phase 2: Explainability Engine

**Project:** ExplainPlan Vision an Explainable Neuro-Symbolic Visual Planning Agent

**Author:** Muhammad Aqib Niazi

**Date:** 2026


Phase 1 gave us a classifier that reaches 96.8% accuracy on 15 plant disease classes.
That accuracy number, on its own, tells us very little about whether the model should
be trusted in practice. A model can be highly accurate on a benchmark while paying
attention to completely the wrong parts of an image, background colour, pot labels,
lighting artefacts rather than the actual disease symptoms.

This phase answers a different question: *why* did the model make each decision, and
*where* in the image is the evidence?

We implement three independent explanation methods that each approach this from a
different angle:

| Method | The question it answers |
|--------|------------------------|
| Grad-CAM++ | Which spatial regions of the leaf drove the classification? |
| SHAP | Which learned visual features contributed most to the score? |
| LIME | Which image segments is the model locally dependent on? |

Using all three matters. When Grad-CAM, SHAP, and LIME all highlight the same region,
the explanation is robust the model is genuinely attending to that area and not
exploiting a shortcut. When they disagree, that disagreement is itself informative: it
signals a case the model is uncertain about and where expert verification is warranted.

Beyond the visual heatmaps, this phase also builds a textual explanation module
natural language descriptions of why the model made its decision and what would change
it. In Phase 5, these text outputs feed into the human-adaptive explanation layer that
adjusts vocabulary and detail level based on who is reading.

### How this phase connects to the rest of the pipeline

```
Phase 1 output
    {disease, confidence, severity, feature_embedding}
                ↓
Phase 2 adds
    {gradcam_heatmap, gradcam_overlay,
     shap_attribution, lime_mask,
     why, why_not, confidence_note, counterfactual}
                ↓
Phase 3 Planning Engine
    reads disease + severity + explanation.why
    to build a sequenced treatment plan
                ↓
Phase 4 Neuro-Symbolic Reasoning
    converts spatial statistics (focus_score, infection_spread)
    into first-order logic facts for the knowledge graph
                ↓
Phase 5 Human-Adaptive Explanation
    selects the right explanation vocabulary
    based on user type (farmer / agronomist / researcher)
```

## Setting Up This Notebook on Kaggle

**Step 1: Save Phase 1 outputs as a Kaggle dataset**

After Phase 1 finishes running, go to your Phase 1 Kaggle notebook. In the right
sidebar, find the Output section and click the folder icon to browse `outputs/`.
Select the entire directory and click **New Dataset**. Name it something like
`explainplan-phase1-outputs`. Kaggle will package everything including
`best_model.pth`, `class_mappings.json`, `config.json`, and the training logs.

**Step 2: Attach the dataset to this notebook**

In this Phase 2 notebook, click **Add data** in the right sidebar and search for
`explainplan-phase1-outputs`. Once attached, the files will be available at
`/kaggle/input/explainplan-phase1-outputs/`. Update `PHASE1_DIR` in the
configuration cell to match that path.

**Step 3: Attach the PlantVillage dataset again**

Phase 2 needs a small sample of real images for LIME and for the comparison figures.
Add the same PlantVillage dataset used in Phase 1.

**A note on compute time**

No training happens in this notebook. We load the frozen Phase 1 model and run
forward passes on a small representative set of images. Grad-CAM and SHAP are fast
(under a second each per image on a T4 GPU). LIME is slower about 20–40 seconds
per image because it runs hundreds of perturbed forward passes. The full notebook
should complete in under 30 minutes of GPU time.


## 1. Install Dependencies

Three XAI libraries are added here on top of the Phase 1 stack:

- `grad-cam` — the pytorch-grad-cam library by Jacob Gildenblat, which supports
  EfficientNet natively and handles hook registration cleanly
- `shap` — provides the GradientExplainer used for feature attribution
- `lime` — provides the image explainer for superpixel-based analysis

Everything else (torch, timm, albumentations) is already present in the Kaggle
environment from Phase 1.

In [ ]:
!pip install -q albumentations timm grad-cam shap lime

## 2. Imports

Imports are grouped by role: standard library, numerical computing, visualisation,
image processing, PyTorch, XAI libraries, and utilities. The grouping makes it easy
to see at a glance which external dependencies this phase introduces on top of Phase 1.

In [ ]:
# Standard library
import os
import json
import random
import warnings
from pathlib import Path

# Numerical
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
from mpl_toolkits.axes_grid1 import make_axes_locatable

# Image processing
import cv2
from PIL import Image

# Progress
from tqdm.auto import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast

# Augmentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Model zoo
import timm

# XAI libraries
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries

# Metrics
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "")

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

## 3. Configuration

Rather than duplicating the Phase 1 hyperparameters by hand, we load them directly
from the saved `config.json`. This guarantees that Phase 2 always uses identical
image dimensions, normalisation statistics, and model architecture to Phase 1
a consistency requirement that is easy to violate accidentally and hard to debug.

Phase 2 only adds its own settings on top: the number of LIME samples, the number
of SHAP background images, the output paths for XAI results.

Update `PHASE1_DIR` to match where your Phase 1 outputs dataset is mounted before
running the notebook.

In [ ]:
# Path to your Phase 1 outputs dataset on Kaggle.
# After saving Phase 1 outputs as a Kaggle dataset and adding it as input,
# the files appear at /kaggle/input/<your-dataset-name>/
PHASE1_DIR = "/kaggle/input/datasets/maqibniazi/explainplan-phase1-outputs"

# PlantVillage dataset — same as Phase 1
DATASET_PATH = "/kaggle/input/datasets/emmarex/plantdisease/PlantVillage"

# Load Phase 1 config — this is the single source of truth for all
# image preprocessing parameters so the two phases stay in sync.
with open(f"{PHASE1_DIR}/outputs/checkpoints/config.json") as f:
    P1_CONFIG = json.load(f)

# Load class mappings
with open(f"{PHASE1_DIR}/outputs/checkpoints/class_mappings.json") as f:
    mappings = json.load(f)

class_to_idx = mappings["class_to_idx"]
idx_to_class = {int(k): v for k, v in mappings["idx_to_class"].items()}
classes      = mappings["classes"]
NUM_CLASSES  = mappings["num_classes"]

# Phase 2 specific settings
CONFIG = {
    # Inherited from Phase 1 — do not change these manually
    "model_name"    : P1_CONFIG["model_name"],
    "image_size"    : P1_CONFIG["image_size"],
    "mean"          : P1_CONFIG["mean"],
    "std"           : P1_CONFIG["std"],
    "embedding_dim" : P1_CONFIG["embedding_dim"],
    "dropout"       : P1_CONFIG["dropout"],
    "device"        : "cuda" if torch.cuda.is_available() else "cpu",

    # Phase 2 specific
    "checkpoint_path"  : f"{PHASE1_DIR}/outputs/checkpoints/best_model.pth",
    "xai_output_dir"   : "outputs/xai",
    "n_demo_images"    : 8,       # images for the comparison figure
    "lime_num_samples" : 1000,    # perturbation samples for LIME
    "lime_num_features": 10,      # superpixel segments to highlight
    "shap_background"  : 50,      # background samples for SHAP DeepExplainer
    "seed"             : 42,
}

# Create output directories
for subdir in ["gradcam", "shap", "lime", "comparison", "counterfactual"]:
    os.makedirs(f"{CONFIG['xai_output_dir']}/{subdir}", exist_ok=True)

print(f"Phase 1 config loaded")
print(f"  Model        : {CONFIG['model_name']}")
print(f"  Image size   : {CONFIG['image_size']}")
print(f"  Classes      : {NUM_CLASSES}")
print(f"  Checkpoint   : {CONFIG['checkpoint_path']}")

## 4. Reproducibility

LIME uses random superpixel sampling and SHAP uses a randomly selected background
distribution. Setting the seed here ensures that every run of this notebook
produces the same explanation outputs a basic requirement for any result that
will appear in a paper or research report.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ["PYTHONHASHSEED"]       = str(seed)

set_seed(CONFIG["seed"])
print(f"Seed set to {CONFIG['seed']}")

## 5. Preprocessing Pipeline

The inference transform is redeclared here from the loaded normalisation constants
rather than hardcoded. This cell is the single source of truth for image preprocessing
in Phase 2. Everything else that needs a preprocessed image calls this transform
nothing else should resize or normalise independently.

The transform is identical to Phase 1 by design. Any deviation would mean the model
receives inputs at test time that differ from what it saw during training, silently
degrading all explanation quality.

In [ ]:
inference_transform = A.Compose([
    A.Resize(CONFIG["image_size"], CONFIG["image_size"]),
    A.Normalize(mean=CONFIG["mean"], std=CONFIG["std"]),
    ToTensorV2()
])

def load_image_rgb(path):
    """Load an image from disk as a uint8 RGB numpy array."""
    img = cv2.imread(path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def preprocess_for_model(image_rgb):
    """
    Apply the inference transform and return a (1, 3, H, W) tensor
    on the configured device.
    """
    tensor = inference_transform(image=image_rgb)["image"]
    return tensor.unsqueeze(0).to(CONFIG["device"])

def preprocess_for_display(image_rgb):
    """
    Resize to model input size but do NOT normalize.
    Used as the RGB float32 base for Grad-CAM overlays.
    """
    resized = cv2.resize(image_rgb, (CONFIG["image_size"], CONFIG["image_size"]))
    return resized.astype(np.float32) / 255.0

print("Preprocessing pipeline ready")

## 6. Model Architecture

The `PlantDiseaseModel` class is copied exactly from Phase 1. This is intentional
and necessary the architecture definition must match byte-for-byte for the Phase 1
weights to load correctly into this notebook.

One thing worth noting: the model is split into two named submodules, `backbone` and
`head`. This separation is what makes Grad-CAM work cleanly. We hook into
`model.backbone.conv_head` the last convolutional layer to extract activation
maps. If the architecture were a flat sequence of layers without this named structure,
identifying the correct target layer would be much more fragile.

In [ ]:
class PlantDiseaseModel(nn.Module):
    """
    Modular perception backbone for ExplainPlan-Vision.

    This definition must remain identical to Phase 1 so that the
    saved checkpoint loads without key mismatches.
    """

    def __init__(self, model_name, num_classes, embedding_dim=512, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=False, num_classes=0)
        backbone_out  = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(backbone_out, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(embedding_dim, num_classes)
        )
        self.embedding_dim = embedding_dim

    def forward(self, x):
        features = self.backbone(x)
        logits   = self.classifier(features)
        return logits, features

    def extract_features(self, x):
        with torch.no_grad():
            return self.backbone(x)


model = PlantDiseaseModel(
    model_name    = CONFIG["model_name"],
    num_classes   = NUM_CLASSES,
    embedding_dim = CONFIG["embedding_dim"],
    dropout       = CONFIG["dropout"]
).to(CONFIG["device"])

model.load_state_dict(
    torch.load(CONFIG["checkpoint_path"],
               map_location=CONFIG["device"],
               weights_only=True)
)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {CONFIG['model_name']}")
print(f"Parameters  : {total_params:,}")
print(f"Mode        : eval (frozen — no training in Phase 2)")

## 7. Building the Representative Image Set

We sample two images per disease class from the dataset and use this fixed set for
all three explanation methods throughout the notebook.

Using the same images for Grad-CAM, SHAP, and LIME is the key decision here. It
means the three-method comparison figure later in the notebook is genuinely
comparable we are not accidentally comparing easy examples for one method against
hard examples for another. Every method faces exactly the same input.

In [ ]:
dataset_path = Path(DATASET_PATH)
set_seed(CONFIG["seed"])

# Collect two random images per class
sample_bank = []  # list of {path, label, plant, disease}

for cls in classes:
    cls_dir = dataset_path / cls
    images  = [f for f in cls_dir.iterdir() if f.suffix.lower() in [".jpg", ".jpeg", ".png"]]
    chosen  = random.sample(images, min(2, len(images)))

    for img_path in chosen:
        if "___" in cls:
            parts   = cls.split("___", 1)
            plant   = parts[0].replace("_", " ").strip()
            disease = parts[1].replace("_", " ").strip()
        else:
            tokens  = cls.split("_")
            plant   = tokens[0].strip()
            disease = " ".join(tokens[1:]).strip()

        sample_bank.append({
            "path"    : str(img_path),
            "label"   : cls,
            "plant"   : plant,
            "disease" : disease,
        })

# Separate demo images (one per class, up to n_demo_images)
demo_images = [sample_bank[i * 2] for i in range(min(CONFIG["n_demo_images"], NUM_CLASSES))]

print(f"Sample bank  : {len(sample_bank)} images across {NUM_CLASSES} classes")
print(f"Demo images  : {len(demo_images)}")

## 8. Inference Function

This is the `predict()` function from Phase 1 with one addition: it now returns
the raw logits tensor alongside the softmax probabilities. SHAP's GradientExplainer
needs the logits rather than the probabilities, because it computes gradients with
respect to the pre-softmax output.

Everything else the output dictionary structure, the severity estimation logic,
the top-3 extraction is unchanged from Phase 1.

In [ ]:
def estimate_severity(confidence, config):
    t = config.get("severity_thresholds",
                   {"high": 0.85, "medium": 0.60, "low": 0.0})
    if confidence >= t["high"]:
        return "high"
    elif confidence >= t["medium"]:
        return "medium"
    return "low"


def predict(image_rgb, model, config):
    """
    Run inference on a uint8 RGB numpy image.

    Returns a dictionary containing the prediction, confidence,
    severity, top-3 predictions, and the raw feature embedding.
    Also returns the logits tensor (on CPU) for downstream XAI use.
    """
    model.eval()
    tensor = preprocess_for_model(image_rgb)

    with torch.no_grad():
        with autocast("cuda", enabled=torch.cuda.is_available()):
            logits, features = model(tensor)
        probs = torch.softmax(logits.float(), dim=1)[0]

    top_idx    = probs.argmax().item()
    confidence = probs[top_idx].item()
    label_name = idx_to_class[top_idx]

    if "___" in label_name:
        parts        = label_name.split("___", 1)
        plant        = parts[0].replace("_", " ").strip()
        disease_type = parts[1].replace("_", " ").strip()
    else:
        tokens       = label_name.split("_")
        plant        = tokens[0].strip()
        disease_type = " ".join(tokens[1:]).strip()

    top3_vals, top3_idxs = torch.topk(probs, k=min(3, NUM_CLASSES))
    top3 = [
        {"class": idx_to_class[i.item()], "confidence": round(v.item(), 4)}
        for v, i in zip(top3_vals, top3_idxs)
    ]

    return {
        "disease"          : label_name,
        "plant"            : plant,
        "disease_type"     : disease_type,
        "confidence"       : round(confidence, 4),
        "severity"         : estimate_severity(confidence, config),
        "top3_predictions" : top3,
        "feature_embedding": features[0].cpu().float().numpy(),
        "is_healthy"       : "healthy" in label_name.lower(),
        "logits_cpu"       : logits.cpu().float()
    }


print("predict() function ready")

## 9. Grad-CAM++ Engine

### Why Grad-CAM++ rather than vanilla Grad-CAM

Grad-CAM (Selvaraju et al., 2017) computes the gradient of the class score with
respect to the final convolutional feature map. Pixels where the gradient is large
are the ones that most influenced the prediction. Grad-CAM++ (Chattopadhay et al.,
2018) extends this with second-order gradient weighting, which produces sharper and
more localised activations particularly important when disease symptoms appear as
small, spatially concentrated lesions rather than broad colour changes across the
whole leaf.

### Which layer to attach to

For EfficientNet-B0, the target layer is `model.backbone.conv_head` the last
convolutional layer before global average pooling collapses the spatial dimensions.
At this point in the network, the feature map is a 7×7 spatial grid (for 224×224
input) encoding high-level semantic patterns. It has enough spatial resolution to
localise which part of the leaf the model focused on, while already capturing the
high-level disease concepts rather than low-level edges.

Attaching earlier in the network would give finer spatial resolution but less
semantically meaningful activations. Attaching later is impossible global pooling
has already destroyed the spatial structure.

### A note on hook management

The `pytorch-grad-cam` library registers backward hooks on the target layer to
capture gradients during the backward pass. These hooks must be removed after each
explanation to prevent state from accumulating across requests. In the backend, this
is handled with a `finally` block to guarantee cleanup even if an exception occurs.

In [ ]:
class GradCAMWrapper(nn.Module):
    """
    Adapter that makes PlantDiseaseModel compatible with pytorch-grad-cam.

    The library expects model(x) to return a plain (batch, num_classes) tensor.
    Our model returns (logits, features), so this wrapper extracts only logits.
    The original model is not modified and continues to expose embeddings for
    SHAP, LIME, and the Phase 3 planning engine.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        logits, _ = self.model(x)
        return logits


class GradCAMEngine:
    """
    Grad-CAM++ wrapper for EfficientNet-B0 loaded via timm.

    Target layer is backbone.conv_head — the last convolutional layer before
    global average pooling. At 224x224 input this produces a 7x7 activation
    map, giving a good balance between spatial resolution and semantic content.
    """

    def __init__(self, model, device):
        self.model         = model
        self.device        = device
        self.wrapped_model = GradCAMWrapper(model).to(device)
        self.wrapped_model.eval()

        # The target layer path goes through the wrapper
        target_layer = [self.wrapped_model.model.backbone.conv_head]

        self.cam = GradCAMPlusPlus(
            model=self.wrapped_model,
            target_layers=target_layer
        )

    def generate(self, image_rgb, target_class_idx=None):
        """
        Generate a Grad-CAM++ heatmap and overlay for the given image.

        Parameters
        ----------
        image_rgb        : uint8 numpy array (H, W, 3)
        target_class_idx : int or None. If None, uses the predicted class.

        Returns
        -------
        heatmap  : float32 numpy array in [0,1], shape (H, W)
        overlay  : uint8 numpy array (H, W, 3), heatmap blended onto image
        """
        tensor    = preprocess_for_model(image_rgb)
        rgb_float = preprocess_for_display(image_rgb)

        targets = [ClassifierOutputTarget(target_class_idx)] if target_class_idx is not None else None

        heatmap = self.cam(input_tensor=tensor, targets=targets)
        heatmap = heatmap[0]  # (H, W)

        overlay = show_cam_on_image(rgb_float, heatmap, use_rgb=True)
        return heatmap, overlay

    def generate_counterfactual(self, image_rgb, runner_up_class_idx):
        """
        Grad-CAM for the runner-up class.
        Answers: what would the model focus on if it believed this was class X?
        """
        return self.generate(image_rgb, target_class_idx=runner_up_class_idx)


gradcam_engine = GradCAMEngine(model, CONFIG["device"])
print("GradCAMEngine ready")
print("  Target layer : wrapped_model.model.backbone.conv_head")

## 10. Generating Grad-CAM Explanations

We run Grad-CAM++ on every image in the representative set and store the results.
Four things are saved per image:

- The original leaf image (resized to 224×224)
- The raw Grad-CAM heatmap as a grayscale float array in [0, 1]
- The overlay the heatmap blended onto the original image with a jet colour map
- The counterfactual overlay the same procedure applied to the runner-up class

The counterfactual overlay is what makes the "why not" explanation visual. It shows
where the model would have looked if it had predicted the second-most-likely disease
instead. Comparing the two overlays gives an immediate intuition for how similar
or different the evidence patterns are between the top two candidate diagnoses.

In [ ]:
gradcam_results = []

for sample in tqdm(demo_images, desc="Grad-CAM"):
    image_rgb  = load_image_rgb(sample["path"])
    prediction = predict(image_rgb, model, CONFIG)

    heatmap, overlay = gradcam_engine.generate(image_rgb)

    # Counterfactual: Grad-CAM for the runner-up class
    runner_up_idx   = class_to_idx[prediction["top3_predictions"][1]["class"]]
    _, cf_overlay   = gradcam_engine.generate_counterfactual(image_rgb, runner_up_idx)

    # Save outputs
    label_safe = sample["label"].replace("/", "_")
    cv2.imwrite(
        f"{CONFIG['xai_output_dir']}/gradcam/{label_safe}_overlay.png",
        cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
    )

    gradcam_results.append({
        "sample"      : sample,
        "prediction"  : prediction,
        "image_rgb"   : image_rgb,
        "heatmap"     : heatmap,
        "overlay"     : overlay,
        "cf_overlay"  : cf_overlay,
        "runner_up"   : prediction["top3_predictions"][1]["class"],
    })

print(f"Grad-CAM generated for {len(gradcam_results)} images")

## 11. Grad-CAM Visualisation Grid

This figure is structured for direct use in a paper or research report. Each row
is one disease class. The three columns are:

1. **Original**: The leaf image as presented to the model
2. **Predicted class activation**: where the model looked for the top-1 prediction
3. **Runner-up class activation**: where the model would have looked for the
   second-most-likely diagnosis

The third column is the most informative for building trust. If the runner-up
activation concentrates on completely different leaf regions than the top-1
activation, the decision is well-grounded the model is using distinct visual
evidence for distinct diagnoses. If the two heatmaps look nearly identical, the
decision boundary between those two classes is spatially ambiguous, and expert
verification is warranted.

In [ ]:
n  = len(gradcam_results)
fig, axes = plt.subplots(n, 3, figsize=(13, n * 3.2))

col_titles = ["Original", "Grad-CAM++ (predicted)", "Grad-CAM++ (runner-up)"]
for j, title in enumerate(col_titles):
    axes[0, j].set_title(title, fontsize=10, fontweight="bold", pad=8)

for i, res in enumerate(gradcam_results):
    original  = cv2.resize(res["image_rgb"], (224, 224))
    pred_name = res["prediction"]["disease_type"]
    conf      = res["prediction"]["confidence"]
    runner_up = res["runner_up"].split("___")[-1].replace("_", " ") if "___" in res["runner_up"] \
                else " ".join(res["runner_up"].split("_")[1:])

    axes[i, 0].imshow(original)
    axes[i, 0].set_ylabel(
        f"{res['sample']['plant']}\n{pred_name}",
        fontsize=7, rotation=0, labelpad=60, va="center"
    )

    axes[i, 1].imshow(res["overlay"])
    axes[i, 1].set_xlabel(f"conf={conf:.3f}", fontsize=7)

    axes[i, 2].imshow(res["cf_overlay"])
    axes[i, 2].set_xlabel(f"if: {runner_up[:25]}", fontsize=7)

    for j in range(3):
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

plt.suptitle(
    "Grad-CAM++ Spatial Explanations — ExplainPlan-Vision Phase 2",
    fontsize=12, fontweight="bold", y=1.005
)
plt.tight_layout()
plt.savefig(f"{CONFIG['xai_output_dir']}/gradcam/gradcam_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print("Grad-CAM grid saved")

## 12. SHAP Feature Attribution Engine

### What SHAP computes

SHAP (Lundberg & Lee, 2017) is rooted in cooperative game theory. It assigns each
input feature a Shapley value a measure of that feature's average marginal
contribution to the prediction across all possible subsets of features. Unlike
Grad-CAM, which is specific to convolutional networks and gradient flow, SHAP is
theoretically grounded in a way that satisfies consistency, local accuracy, and
missingness axioms simultaneously.

### What we attribute to

Rather than computing Shapley values over raw pixels which is extremely slow for
high-resolution images we apply SHAP to the 1280-dimensional feature embedding
produced by EfficientNet's backbone before the classifier head. This tells us which
of the model's learned visual concepts drove the prediction. The attribution is then
projected back onto the original image using the model's internal spatial structure.

This approach is both faster and more interpretable than pixel-level SHAP. The
embedding dimensions correspond to learned visual patterns (textures, shapes, colour
gradients) rather than individual pixels, which have no inherent semantic meaning.

### Background sample selection

SHAP's GradientExplainer requires a background dataset a reference that represents
"what the model sees when given an uninformative input". We use 50 randomly sampled
images from across all disease classes. A diverse background ensures that attributions
measure deviation from the global average disease appearance, not from a class-specific
baseline that would bias the results.

In [ ]:
def disable_inplace_activations(model):
    """
    Set inplace=False on all activation modules recursively.
    Required because SHAP's backward hooks conflict with SiLU(inplace=True)
    in EfficientNet, causing 'view is being modified inplace' RuntimeError.
    """
    for module in model.modules():
        if hasattr(module, "inplace"):
            module.inplace = False


class SHAPModelWrapper(nn.Module):
    """
    nn.Module wrapper so shap.GradientExplainer receives a proper module.
    Extracts only logits from the (logits, features) tuple our model returns.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        logits, _ = self.model(x)
        return logits


class SHAPEngine:
    """
    SHAP GradientExplainer for pixel-level attribution on EfficientNet.

    GradientExplainer is used instead of DeepExplainer because EfficientNet's
    Squeeze-and-Excitation blocks and depthwise separable convolutions violate
    the linear decomposition assumption that DeepExplainer requires. This
    shows up as a large additivity error (max diff ~5.0 vs tolerance 0.01).

    GradientExplainer works by sampling random convex combinations of the input
    and a set of background images, then computing gradients at each sample.
    It is compatible with arbitrary architectures and produces stable
    pixel-level attributions.
    """

    def __init__(self, model, background_paths, device, n_samples=50):
        self.device = device

        # Must disable inplace activations before any SHAP hooks are installed
        disable_inplace_activations(model)
        print("In-place activations disabled")

        self.shap_model = SHAPModelWrapper(model).to(device)
        self.shap_model.eval()

        # Build background tensor on the same device as the model.
        # GradientExplainer uses these as interpolation anchors, not a
        # distribution expectation, so 30-50 diverse images is sufficient.
        print(f"Building SHAP background from {n_samples} images...")
        bg_tensors = []
        chosen = random.sample(background_paths, min(n_samples, len(background_paths)))
        for path in tqdm(chosen, desc="Background", leave=False):
            img    = load_image_rgb(path)
            tensor = inference_transform(image=img)["image"]  # (3, H, W) CPU float
            bg_tensors.append(tensor.unsqueeze(0))

        # GradientExplainer keeps the background on the same device as the model
        background = torch.cat(bg_tensors, dim=0).float().to(device)
        print(f"Background tensor: {background.shape}")

        # GradientExplainer signature: (model, data)
        # data can be a tensor directly — no additivity constraint
        self.explainer  = shap.GradientExplainer(self.shap_model, background)
        self.n_bg       = background.shape[0]
        print("SHAP GradientExplainer ready")

    def explain(self, image_rgb, top_class_idx):
        """
        Compute SHAP gradient attribution for the predicted class.

        Returns
        -------
        class_shap   : numpy array (3, H, W) per-channel attribution
        shap_display : numpy array (H, W) mean absolute attribution
                       normalized to [0, 1]
        """
        H = W = CONFIG["image_size"]

        tensor = inference_transform(image=image_rgb)["image"]
        tensor = tensor.unsqueeze(0).float().to(self.device)

        shap_values, indices = self.explainer.shap_values(
            tensor,
            ranked_outputs=1,
            nsamples=50
        )

        # shap_values[0] shape varies across SHAP versions:
        # (1, 3, H, W), (1, 3, H, 1), (3, H, W), (3, H, 1), etc.
        # We normalize to (3, H, W) unconditionally.
        raw = np.array(shap_values[0])  # force numpy
        raw = raw.squeeze()             # remove all size-1 dims

        # After squeeze: could be (3, H, W), (H, W, 3), (H,), etc.
        # Handle each case explicitly.
        if raw.ndim == 3:
            if raw.shape[0] == 3:
                # Already (3, H, W)
                class_shap = raw
            elif raw.shape[2] == 3:
                # Channel-last (H, W, 3) — transpose to (3, H, W)
                class_shap = raw.transpose(2, 0, 1)
            else:
                # Unknown 3D layout — use mean across first axis
                class_shap = np.stack([raw] * 3, axis=0)
        elif raw.ndim == 2:
            # Single channel (H, W) — replicate to (3, H, W)
            class_shap = np.stack([raw] * 3, axis=0)
        elif raw.ndim == 1:
            # Fully collapsed — reshape to (3, H, W)
            total = raw.size
            if total == 3 * H * W:
                class_shap = raw.reshape(3, H, W)
            else:
                class_shap = np.stack([raw.reshape(H, W)] * 3, axis=0)
        else:
            raise ValueError(f"Cannot handle shap_values shape: {raw.shape}")

        # Ensure class_shap is exactly (3, H, W)
        if class_shap.shape != (3, H, W):
            class_shap = np.resize(class_shap, (3, H, W))

        shap_display = np.mean(np.abs(class_shap), axis=0)   # (H, W)
        assert shap_display.shape == (H, W), \
            f"shap_display still wrong: {shap_display.shape}"

        lo, hi = shap_display.min(), shap_display.max()
        shap_display = (shap_display - lo) / (hi - lo + 1e-8)

        return class_shap, shap_display

# Collect all image paths for background sampling
all_paths = []
for cls in classes:
    cls_dir = dataset_path / cls
    all_paths.extend([
        str(f) for f in cls_dir.iterdir()
        if f.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ])

print(f"Background pool: {len(all_paths)} images across {NUM_CLASSES} classes")

shap_engine = SHAPEngine(
    model,
    all_paths,
    CONFIG["device"],
    n_samples=CONFIG["shap_background"]
)

## 13. Computing SHAP Explanations

We run the SHAP GradientExplainer on each image in the representative set, using the
background distribution built in the previous cell.

One engineering detail worth noting: SHAP's GradientExplainer and Grad-CAM's backward
hooks conflict when both are active simultaneously on the same model. The Grad-CAM
hooks must be fully removed before initialising the SHAP explainer. This is why the
SHAP engine is initialised after the Grad-CAM loop completes the ordering is not
arbitrary.

In [ ]:
shap_results = []

for res in tqdm(gradcam_results, desc="SHAP"):
    image_rgb  = res["image_rgb"]
    prediction = res["prediction"]
    top_idx    = class_to_idx[prediction["disease"]]

    class_shap, shap_display = shap_engine.explain(image_rgb, top_idx)

    shap_results.append({
        "sample"       : res["sample"],
        "prediction"   : prediction,
        "image_rgb"    : image_rgb,
        "class_shap"   : class_shap,
        "shap_display" : shap_display,
    })

print(f"SHAP attributions computed for {len(shap_results)} images")

## 14. SHAP Attribution Grid

The SHAP attribution map uses a diverging colour scale: red regions are pixels that
pushed the prediction *toward* the predicted class, blue regions pushed it *away*.
Neutral pixels (grey) had no meaningful contribution.

For a well-behaved disease classifier, the red regions should correspond to the
visible disease symptoms on the leaf lesions, spots, discolouration and the blue
regions should correspond to healthy leaf tissue or background. When this pattern
holds across multiple images of the same disease class, it provides strong evidence
that the model has learned the correct visual features rather than spurious correlations.

In [ ]:
n   = len(shap_results)
fig, axes = plt.subplots(n, 3, figsize=(13, n * 3.2))

col_titles = ["Original", "SHAP attribution", "SHAP overlay"]
for j, title in enumerate(col_titles):
    axes[0, j].set_title(title, fontsize=10, fontweight="bold", pad=8)

for i, res in enumerate(shap_results):
    original     = cv2.resize(res["image_rgb"], (224, 224))
    shap_display = res["shap_display"]
    conf         = res["prediction"]["confidence"]
    disease      = res["prediction"]["disease_type"]

    # SHAP overlay: apply colormap and blend
    shap_colored = cm.RdBu_r(shap_display)[:, :, :3]
    orig_float   = original.astype(np.float32) / 255.0
    shap_overlay = (0.55 * orig_float + 0.45 * shap_colored).clip(0, 1)

    axes[i, 0].imshow(original)
    axes[i, 0].set_ylabel(
        f"{res['sample']['plant']}\n{disease}",
        fontsize=7, rotation=0, labelpad=60, va="center"
    )

    im = axes[i, 1].imshow(shap_display, cmap="RdBu_r", vmin=0, vmax=1)
    axes[i, 1].set_xlabel(f"conf={conf:.3f}", fontsize=7)

    axes[i, 2].imshow(shap_overlay)

    for j in range(3):
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

plt.suptitle(
    "SHAP Pixel Attributions — ExplainPlan-Vision Phase 2",
    fontsize=12, fontweight="bold", y=1.005
)
plt.tight_layout()
plt.savefig(f"{CONFIG['xai_output_dir']}/shap/shap_grid.png", dpi=150, bbox_inches="tight")
plt.show()
print("SHAP grid saved")

## 15. LIME Superpixel Explanation Engine

### How LIME works

LIME (Ribeiro et al., 2016) takes a fundamentally different approach to explanation
than Grad-CAM or SHAP. Rather than probing the model's internal gradients, LIME
treats the model as a complete black box and probes it through its input-output
behaviour.

The process for one image:

1. Segment the image into superpixels semantically coherent regions of similar
   colour and texture
2. Generate hundreds of perturbed versions of the image by randomly masking
   (graying out) different subsets of superpixels
3. Run each perturbed image through the model and record the prediction confidence
4. Fit a simple linear model to these (perturbation, confidence) pairs
5. The linear model's coefficients tell us which superpixels contributed most
   positively or negatively to the prediction

### Why this complements Grad-CAM and SHAP

Grad-CAM depends on gradient flow through the network it can only highlight
regions that the network's backward pass passes signal through. SHAP depends on
the structure of the feature embedding. LIME is completely independent of both.

When all three methods agree on the important regions, that convergence is much
stronger evidence than any one method alone. The quantitative agreement analysis
in Cell 23 makes this convergence measurable rather than just visually inspected.

### A practical note on speed

LIME is the slowest of the three methods by a significant margin. Each image
requires 500–1000 perturbed forward passes. On a T4 GPU this takes 20–40 seconds
per image. This is why LIME is marked as optional in the production backend it
is too slow for real-time inference at the deployed API endpoint.

In [ ]:
class LIMEEngine:
    """
    LIME image explainer wrapping the plant disease model.

    LIME requires a prediction function that accepts a batch of uint8
    numpy images (N, H, W, 3) and returns a probability matrix (N, C).
    """

    def __init__(self, model, device, num_samples=1000):
        self.model       = model
        self.device      = device
        self.num_samples = num_samples
        self.explainer   = lime_image.LimeImageExplainer()

    def _batch_predict(self, images):
        """
        Prediction function for LIME.
        Accepts (N, H, W, 3) uint8 numpy array, returns (N, C) float32.
        """
        self.model.eval()
        batch_tensors = []
        for img in images:
            t = inference_transform(image=img)["image"]
            batch_tensors.append(t)

        batch = torch.stack(batch_tensors).to(self.device)

        with torch.no_grad():
            logits, _ = self.model(batch)
            probs     = torch.softmax(logits.float(), dim=1)

        return probs.cpu().numpy()

    def explain(self, image_rgb, top_class_idx, num_features=10):
        """
        Compute LIME explanation for the predicted class.

        Parameters
        ----------
        image_rgb      : uint8 numpy array (H, W, 3)
        top_class_idx  : int — the class to explain
        num_features   : int — number of superpixels to highlight

        Returns
        -------
        explanation    : LIME Explanation object
        lime_mask      : (H, W, 3) uint8 overlay image
        """
        # LIME expects the image at its natural resolution; it handles
        # resizing internally via the prediction function wrapper.
        explanation = self.explainer.explain_instance(
            image_rgb,
            self._batch_predict,
            top_labels=3,
            hide_color=0,
            num_samples=self.num_samples
        )

        temp, mask = explanation.get_image_and_mask(
            top_class_idx,
            positive_only=True,
            num_features=num_features,
            hide_rest=False
        )

        lime_overlay = mark_boundaries(temp / 255.0, mask)
        lime_overlay = (lime_overlay * 255).astype(np.uint8)

        return explanation, lime_overlay, mask


lime_engine = LIMEEngine(
    model,
    CONFIG["device"],
    num_samples=CONFIG["lime_num_samples"]
)
print("LIMEEngine ready")

## 16. Computing LIME Explanations

LIME runs on the same representative image set used for Grad-CAM and SHAP. The
`num_samples` parameter controls how many perturbed images are generated per
explanation. Higher values give more stable results but take proportionally longer.
The default of 1000 samples is a reasonable trade-off for research quality output.

The output for each image is a binary mask showing which superpixels had positive
contribution (shown in the original colour) and which had negative or neutral
contribution (grayed out).

In [ ]:
lime_results = []

for res in tqdm(gradcam_results, desc="LIME"):
    image_rgb  = res["image_rgb"]
    prediction = res["prediction"]
    top_idx    = class_to_idx[prediction["disease"]]

    explanation, lime_overlay, mask = lime_engine.explain(
        image_rgb, top_idx, num_features=CONFIG["lime_num_features"]
    )

    label_safe = res["sample"]["label"].replace("/", "_")
    cv2.imwrite(
        f"{CONFIG['xai_output_dir']}/lime/{label_safe}_lime.png",
        cv2.cvtColor(lime_overlay, cv2.COLOR_RGB2BGR)
    )

    lime_results.append({
        "sample"       : res["sample"],
        "prediction"   : prediction,
        "image_rgb"    : image_rgb,
        "lime_overlay" : lime_overlay,
        "mask"         : mask,
    })

print(f"LIME explanations computed for {len(lime_results)} images")

## 17. Three-Method Comparison Figure

This is the central research figure of Phase 2. It puts all three explanation
methods side by side for the same images so the reader can directly assess how
well they agree.

Each row is one disease class. The four columns are:

1. **Original** — the unmodified leaf image
2. **Grad-CAM++** — spatial activation heatmap
3. **SHAP** — pixel attribution (red = supports prediction)
4. **LIME** — superpixel importance mask

When all three highlight the same regions which they typically do for high-confidence
predictions of visually distinctive diseases the explanation is robust. When they
diverge, the divergence reveals something genuinely interesting about how the model
is processing that particular image, and is worth examining closely rather than treating
as an inconvenience.

This figure is designed at a resolution and layout suitable for direct inclusion in a
conference paper.

In [ ]:
n   = len(gradcam_results)
fig, axes = plt.subplots(n, 4, figsize=(17, n * 3.2))

col_titles = ["Original", "Grad-CAM++", "SHAP", "LIME"]
for j, title in enumerate(col_titles):
    axes[0, j].set_title(title, fontsize=11, fontweight="bold", pad=10)

for i in range(n):
    gc_res   = gradcam_results[i]
    sh_res   = shap_results[i]
    li_res   = lime_results[i]

    original  = cv2.resize(gc_res["image_rgb"], (224, 224))
    pred      = gc_res["prediction"]
    disease   = pred["disease_type"]
    conf      = pred["confidence"]
    plant     = gc_res["sample"]["plant"]

    # Shap overlay
    shap_colored = cm.RdBu_r(sh_res["shap_display"])[:, :, :3]
    orig_float   = original.astype(np.float32) / 255.0
    shap_overlay = (0.55 * orig_float + 0.45 * shap_colored).clip(0, 1)

    axes[i, 0].imshow(original)
    axes[i, 0].set_ylabel(
        f"{plant}\n{disease[:20]}\nconf={conf:.3f}",
        fontsize=7.5, rotation=0, labelpad=75, va="center"
    )

    axes[i, 1].imshow(gc_res["overlay"])
    axes[i, 2].imshow(shap_overlay)
    axes[i, 3].imshow(li_res["lime_overlay"])

    for j in range(4):
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

plt.suptitle(
    "Three-Method XAI Comparison — ExplainPlan-Vision Phase 2\n"
    "Left to right: original / Grad-CAM++ / SHAP / LIME",
    fontsize=12, fontweight="bold", y=1.005
)
plt.tight_layout()
plt.savefig(
    f"{CONFIG['xai_output_dir']}/comparison/three_method_comparison.png",
    dpi=150, bbox_inches="tight"
)
plt.show()
print("Three-method comparison figure saved")

## 18. Natural Language Explanation Generator

Visual heatmaps communicate with researchers and technically literate users. A farmer
in the field needs language.

This module generates four types of natural language explanation for any prediction,
grounded entirely in the model's output statistics and the Grad-CAM spatial analysis
no LLM is involved at this stage. The explanations are rule-based templates that
draw on a small disease knowledge base encoding pathogen type, visual cues, and
spread characteristics for each disease class.

The four explanation types:

| Type | What it says |
|------|-------------|
| `why` | Why the model predicted this disease citing the visual evidence and pathogen characteristics |
| `why_not` | Why the runner-up disease was rejected based on the confidence gap and differing visual cues |
| `confidence` | A plain-language interpretation of the confidence score |
| `counterfactual` | What would have to change in the image for the prediction to shift |

In Phase 5, these template-generated strings will be extended with a grounded LLM
layer that adapts vocabulary and technical depth to the identified user type keeping
all the factual grounding from this phase while making the language more natural and
audience-appropriate.

In [ ]:
# Disease-specific knowledge base for textual explanations.
# This will grow substantially in Phase 3 when the planning engine is added.
DISEASE_KNOWLEDGE = {
    "Early blight": {
        "pathogen"    : "fungal (Alternaria solani)",
        "visual_cues" : "dark concentric rings on older leaves",
        "spread"      : "spore dispersal via wind and water splash",
    },
    "Late blight": {
        "pathogen"    : "oomycete (Phytophthora infestans)",
        "visual_cues" : "water-soaked lesions with pale green border",
        "spread"      : "rapid in cool, wet conditions",
    },
    "Bacterial spot": {
        "pathogen"    : "bacterial (Xanthomonas spp.)",
        "visual_cues" : "small water-soaked spots with yellow halo",
        "spread"      : "rain splash and infected transplants",
    },
    "Septoria leaf spot": {
        "pathogen"    : "fungal (Septoria lycopersici)",
        "visual_cues" : "small circular spots with dark borders and grey centres",
        "spread"      : "splashing water and infected plant debris",
    },
    "Leaf Mold": {
        "pathogen"    : "fungal (Passalora fulva)",
        "visual_cues" : "pale greenish-yellow spots above, olive-green mould below",
        "spread"      : "high humidity and poor air circulation",
    },
    "Target Spot": {
        "pathogen"    : "fungal (Corynespora cassiicola)",
        "visual_cues" : "brown spots with concentric rings resembling a target",
        "spread"      : "warm wet conditions",
    },
    "Spider mites Two spotted spider mite": {
        "pathogen"    : "arthropod pest (Tetranychus urticae)",
        "visual_cues" : "stippling, bronzing, and fine webbing on leaves",
        "spread"      : "wind, contact, and contaminated equipment",
    },
    "Tomato YellowLeaf  Curl Virus": {
        "pathogen"    : "viral (TYLCV)",
        "visual_cues" : "upward leaf curl and yellowing of margins",
        "spread"      : "whitefly (Bemisia tabaci) vector",
    },
    "Tomato mosaic virus": {
        "pathogen"    : "viral (ToMV)",
        "visual_cues" : "mosaic discolouration and leaf distortion",
        "spread"      : "mechanical contact and contaminated tools",
    },
}


def generate_explanation(prediction, gradcam_heatmap):
    """
    Generate four types of natural language explanation for a prediction.

    Returns a dictionary with:
        why          - why this class was predicted
        why_not      - why the runner-up was rejected
        confidence   - interpretation of the confidence score
        counterfactual - what would change the prediction
    """
    disease       = prediction["disease_type"]
    confidence    = prediction["confidence"]
    severity      = prediction["severity"]
    top3          = prediction["top3_predictions"]
    runner_up     = top3[1]["class"] if len(top3) > 1 else None
    runner_up_conf= top3[1]["confidence"] if len(top3) > 1 else 0.0
    conf_gap      = round(confidence - runner_up_conf, 4)
    is_healthy    = prediction["is_healthy"]

    # Determine how much of the heatmap is concentrated
    # in the top-20% of activation values (focus score)
    threshold    = np.percentile(gradcam_heatmap, 80)
    focus_ratio  = float((gradcam_heatmap >= threshold).mean())
    focus_desc   = "localised" if focus_ratio < 0.08 else "distributed"

    # Retrieve known disease characteristics
    knowledge = DISEASE_KNOWLEDGE.get(disease, {})
    visual_cue = knowledge.get("visual_cues", "symptom patterns on the leaf surface")
    pathogen   = knowledge.get("pathogen", "an identified pathogen")

    if is_healthy:
        why_text = (
            f"The leaf appears healthy. The model found no visual patterns "
            f"consistent with any of the {NUM_CLASSES - 1} disease classes "
            f"in its training distribution. Confidence: {confidence:.1%}."
        )
    else:
        why_text = (
            f"The model predicted {disease} ({pathogen}) with {confidence:.1%} confidence. "
            f"The decision was driven by {focus_desc} activation regions consistent "
            f"with {visual_cue}. The severity is assessed as {severity} based on "
            f"the prediction confidence."
        )

    # Why-not explanation
    if runner_up and not is_healthy:
        ru_disease = runner_up.split("___")[-1].replace("_", " ") if "___" in runner_up \
                     else " ".join(runner_up.split("_")[1:])
        ru_knowledge = DISEASE_KNOWLEDGE.get(ru_disease, {})
        ru_cue       = ru_knowledge.get("visual_cues", "different symptom patterns")
        why_not_text = (
            f"{ru_disease} was considered but rejected. Its characteristic features "
            f"({ru_cue}) were not prominently present in this image. "
            f"The confidence gap between the two predictions was {conf_gap:.1%}."
        )
    else:
        why_not_text = "No disease classes had significant competing activation."

    # Confidence interpretation
    if confidence >= 0.90:
        conf_text = (
            f"Confidence is high ({confidence:.1%}). The visual evidence in this "
            f"image strongly matches the learned pattern for {disease}."
        )
    elif confidence >= 0.70:
        conf_text = (
            f"Confidence is moderate ({confidence:.1%}). The system recommends "
            f"verifying this diagnosis with a second image or expert consultation."
        )
    else:
        conf_text = (
            f"Confidence is low ({confidence:.1%}). Treat this prediction with "
            f"caution. Image quality or unusual symptom presentation may be a factor."
        )

    # Counterfactual
    counterfactual_text = (
        f"If the {focus_desc} lesion regions were absent or less pronounced, "
        f"the prediction would likely shift toward "
        f"{runner_up.split('___')[-1].replace('_', ' ') if runner_up and '___' in runner_up else 'a related condition'} "
        f"or Healthy. Reducing image noise and ensuring consistent lighting "
        f"would improve prediction stability."
    )

    return {
        "why"            : why_text,
        "why_not"        : why_not_text,
        "confidence"     : conf_text,
        "counterfactual" : counterfactual_text,
    }


print("generate_explanation() ready")

## 19. Textual Explanation Demo

We run the explanation generator on the first three images in the representative set
and print the output. The goal here is qualitative review reading these explanations
and asking whether they make sense agronomically, not whether they pass a unit test.

A few things to check when reading the output:
- Does the *why* explanation correctly identify the visual cue that distinguishes
  this disease from similar ones?
- Is the *why_not* explanation plausible does the runner-up really have different
  visual characteristics from the predicted class?
- Does the confidence interpretation match what the number actually means clinically?

In [ ]:
for res in gradcam_results[:3]:
    explanation = generate_explanation(res["prediction"], res["heatmap"])
    print("=" * 60)
    print(f"Image  : {res['sample']['plant']} / {res['prediction']['disease_type']}")
    print(f"Conf   : {res['prediction']['confidence']:.3f}  Severity: {res['prediction']['severity']}")
    print()
    print("WHY")
    print(explanation["why"])
    print()
    print("WHY NOT")
    print(explanation["why_not"])
    print()
    print("CONFIDENCE")
    print(explanation["confidence"])
    print()
    print("COUNTERFACTUAL")
    print(explanation["counterfactual"])
    print()

## 20. The Unified explain() Interface

This function is the public interface that all downstream phases consume. It takes a
raw image path, runs the full explanation pipeline, and returns a single structured
dictionary.

The interface contract with Phase 3 is defined by which fields Phase 3 actually reads:

| Field | Used by Phase 3 for |
|-------|-------------------|
| `prediction.disease` | Knowledge base lookup to select treatment rules |
| `prediction.severity` | Determining treatment urgency |
| `prediction.confidence` | Confidence gating — plans are flagged if confidence < 0.70 |
| `prediction.plant` | Crop-specific rule selection |
| `prediction.is_healthy` | Early exit — no treatment plan generated for healthy plants |
| `explanation.why` | Included as rationale in the treatment plan output |
| `explanation.why_not` | Included in the alternative plan for the runner-up diagnosis |

The `run_lime` parameter defaults to `True` in this notebook but is set to `False`
in the deployed backend, where LIME's 20–40 second runtime per image is not
acceptable for a synchronous API call.

In [ ]:
def explain(image_path, model, config, run_lime=True):
    """
    Full explainability pipeline for a single image.

    Parameters
    ----------
    image_path : str — path to input image
    model      : PlantDiseaseModel
    config     : CONFIG dict
    run_lime   : bool — LIME is slow; set False for real-time use

    Returns
    -------
    A dictionary with:
        prediction       - full Phase 1 prediction dict
        gradcam_heatmap  - (H, W) float32 array
        gradcam_overlay  - (H, W, 3) uint8 RGB
        shap_display     - (H, W) float32 attribution map
        lime_overlay     - (H, W, 3) uint8 RGB (or None if run_lime=False)
        explanation      - dict with why/why_not/confidence/counterfactual
        image_rgb        - original image
    """
    image_rgb  = load_image_rgb(image_path)
    prediction = predict(image_rgb, model, config)
    top_idx    = class_to_idx[prediction["disease"]]

    # Grad-CAM
    heatmap, gc_overlay = gradcam_engine.generate(image_rgb)

    # SHAP
    _, shap_display = shap_engine.explain(image_rgb, top_idx)

    # LIME (optional)
    lime_overlay = None
    if run_lime:
        _, lime_overlay, _ = lime_engine.explain(
            image_rgb, top_idx, num_features=config["lime_num_features"]
        )

    # Textual explanations
    text_explanation = generate_explanation(prediction, heatmap)

    return {
        "prediction"      : prediction,
        "gradcam_heatmap" : heatmap,
        "gradcam_overlay" : gc_overlay,
        "shap_display"    : shap_display,
        "lime_overlay"    : lime_overlay,
        "explanation"     : text_explanation,
        "image_rgb"       : image_rgb,
    }


print("explain() unified interface ready")
print()
print("Output schema for Phase 3:")
print("  explain(path) -> {")
print("    prediction: {disease, confidence, severity, feature_embedding, ...}")
print("    gradcam_overlay: (224, 224, 3) uint8")
print("    shap_display:    (224, 224) float32")
print("    lime_overlay:    (H, W, 3) uint8")
print("    explanation: {why, why_not, confidence, counterfactual}")
print("  }")

## 21. Embedding-Based Counterfactual Retrieval

A counterfactual example in this context is a real photograph from the dataset that
is as visually similar as possible to the query image, but belongs to a different
disease class.

Showing a farmer a real photograph of the alternative disease is more grounded and
convincing than any text description. It makes the *why not* explanation concrete:
"The model considered Late Blight, and here is what Late Blight actually looks like
on a tomato leaf  you can see the water-soaked margins that are absent in your image."

The retrieval works by computing cosine similarity between the query image's
1280-dimensional feature embedding and the embeddings of all images in a reference
index built from 30 images per class. We return the nearest neighbour from each of
the top-2 alternative classes.

Cosine similarity in embedding space is the right metric here because the backbone
was trained to produce embeddings where disease classes cluster together. Two images
with similar embeddings will look visually similar to the model, regardless of their
raw pixel values.

In [ ]:
# Build an embedding index over a representative sample of the full dataset.
# We sample 30 images per class to keep memory manageable.

INDEX_SAMPLE_PER_CLASS = 30
index_records = []

for cls in tqdm(classes, desc="Building embedding index"):
    cls_dir = dataset_path / cls
    imgs    = [str(f) for f in cls_dir.iterdir()
               if f.suffix.lower() in [".jpg", ".jpeg", ".png"]]
    chosen  = random.sample(imgs, min(INDEX_SAMPLE_PER_CLASS, len(imgs)))

    model.eval()
    with torch.no_grad():
        for path in chosen:
            img    = load_image_rgb(path)
            tensor = preprocess_for_model(img)
            feat   = model.extract_features(tensor)[0].cpu().float().numpy()
            index_records.append({"path": path, "label": cls, "embedding": feat})

index_df         = pd.DataFrame(index_records)
embedding_matrix = np.stack(index_df["embedding"].values)  # (N, 1280)

print(f"Embedding index built: {len(index_df)} images")


def retrieve_counterfactual(query_embedding, query_class, top_k=1):
    """
    Find the most similar image from a different class.

    Returns a list of {path, label, similarity} dicts.
    """
    sims   = cosine_similarity(query_embedding.reshape(1, -1), embedding_matrix)[0]
    # Mask same class
    is_other = index_df["label"].values != query_class
    sims_filtered = np.where(is_other, sims, -1)
    top_indices   = np.argsort(sims_filtered)[::-1][:top_k]

    return [
        {
            "path"       : index_df.iloc[idx]["path"],
            "label"      : index_df.iloc[idx]["label"],
            "similarity" : round(float(sims[idx]), 4)
        }
        for idx in top_indices
    ]


print("retrieve_counterfactual() ready")

## 22. Counterfactual Retrieval Demo

For each of the first four demo images, we display the query image alongside its
nearest counterfactual neighbour the most visually similar image that the model
classifies as a different disease. The disease labels, confidence scores, and
embedding similarity are shown below each image.

Look for cases where the query and counterfactual are genuinely visually similar
these are the cases where the model's decision is most interesting and the
counterfactual explanation most informative.

In [ ]:
n   = min(4, len(gradcam_results))
fig, axes = plt.subplots(n, 2, figsize=(9, n * 3.5))

axes[0, 0].set_title("Query image",         fontsize=10, fontweight="bold")
axes[0, 1].set_title("Counterfactual (nearest different class)", fontsize=10, fontweight="bold")

for i in range(n):
    res      = gradcam_results[i]
    pred     = res["prediction"]
    emb      = pred["feature_embedding"]
    cf_list  = retrieve_counterfactual(emb, pred["disease"], top_k=1)

    query_img = cv2.resize(res["image_rgb"], (224, 224))
    cf_img    = load_image_rgb(cf_list[0]["path"])
    cf_img    = cv2.resize(cf_img, (224, 224))

    cf_label  = cf_list[0]["label"].split("___")[-1].replace("_", " ") \
                if "___" in cf_list[0]["label"] \
                else " ".join(cf_list[0]["label"].split("_")[1:])

    axes[i, 0].imshow(query_img)
    axes[i, 0].set_ylabel(
        f"{pred['plant']}\n{pred['disease_type']}",
        fontsize=8, rotation=0, labelpad=75, va="center"
    )
    axes[i, 0].set_xlabel(f"conf={pred['confidence']:.3f}", fontsize=8)

    axes[i, 1].imshow(cf_img)
    axes[i, 1].set_xlabel(f"{cf_label[:30]}  (sim={cf_list[0]['similarity']:.3f})", fontsize=8)

    for j in range(2):
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

plt.suptitle(
    "Counterfactual Retrieval — Most Similar Image from Different Class",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.savefig(f"{CONFIG['xai_output_dir']}/counterfactual/counterfactual_pairs.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Counterfactual figure saved")

## 23. Quantitative Method Agreement Analysis

We compute Spearman rank correlation between the Grad-CAM heatmap and the SHAP
attribution map for each image. Spearman correlation is appropriate here because
we care about the ranking of pixel importance, not the absolute magnitude values
(which are on different scales for the two methods).

High correlation means both methods point to the same pixels as important.
Low correlation is a flag — it means the two methods are giving contradictory
explanations, which warrants closer investigation. For a well-trained model on
distinctive disease classes, Grad-CAM and SHAP should agree most of the time.

This table can go directly into a paper's explainability evaluation section.
It gives a quantitative basis for the claim that the explanations are robust,
rather than relying solely on the visual impression from the comparison figures.

In [ ]:
from scipy.stats import spearmanr

agreement_rows = []

for gc_res, sh_res in zip(gradcam_results, shap_results):
    heatmap      = gc_res["heatmap"].flatten()
    shap_flat    = sh_res["shap_display"].flatten()
    corr, pvalue = spearmanr(heatmap, shap_flat)

    agreement_rows.append({
        "disease"              : gc_res["prediction"]["disease_type"],
        "confidence"           : gc_res["prediction"]["confidence"],
        "spearman_gradcam_shap": round(corr, 4),
        "p_value"              : round(pvalue, 6),
        "significant"          : pvalue < 0.05
    })

agreement_df = pd.DataFrame(agreement_rows)
print("XAI Method Agreement (Grad-CAM vs SHAP, Spearman rank correlation)")
print(agreement_df.to_string(index=False))
print()
print(f"Mean Spearman r : {agreement_df['spearman_gradcam_shap'].mean():.4f}")
agreement_df.to_csv(f"{CONFIG['xai_output_dir']}/comparison/method_agreement.csv", index=False)

## 24. Saving Phase 2 Outputs for Phase 3

Everything Phase 3 needs is saved to `outputs/xai/`:

- `explain_schema.json` — the interface contract between Phase 2 and Phase 3,
  documenting exactly what the `explain()` function returns and which fields
  Phase 3 is expected to consume
- `method_agreement.csv` — the Spearman correlation table from Cell 23
- Individual heatmaps and overlays for each demo image

The `explain_schema.json` file is the most important output. It is the contract.
Any change to the `explain()` function's output dictionary must be reflected in
this schema file, otherwise Phase 3 will silently receive unexpected data and fail
in ways that are difficult to debug.

After this cell runs, save the entire `outputs/xai/` directory as a Kaggle dataset
named `explainplan-phase2-outputs` and add it as an input to the Phase 3 notebook.

In [ ]:
# Document the explain() output schema for Phase 3
explain_schema = {
    "description": "Output schema of the explain() function in Phase 2",
    "fields": {
        "prediction": {
            "disease"          : "full class label string",
            "plant"            : "plant type parsed from label",
            "disease_type"     : "disease name parsed from label",
            "confidence"       : "float in [0,1]",
            "severity"         : "high | medium | low",
            "top3_predictions" : "list of {class, confidence}",
            "feature_embedding": "numpy array shape (1280,)",
            "is_healthy"       : "boolean"
        },
        "gradcam_heatmap"  : "numpy float32 (224, 224) in [0,1]",
        "gradcam_overlay"  : "numpy uint8 (224, 224, 3)",
        "shap_display"     : "numpy float32 (224, 224) in [0,1]",
        "lime_overlay"     : "numpy uint8 (H, W, 3) or None",
        "explanation": {
            "why"            : "string — why this class was predicted",
            "why_not"        : "string — why runner-up was rejected",
            "confidence"     : "string — confidence interpretation",
            "counterfactual" : "string — what would change the prediction"
        }
    },
    "phase3_fields_used": ["prediction.disease", "prediction.severity",
                           "prediction.confidence", "prediction.is_healthy",
                           "prediction.plant", "explanation.why",
                           "explanation.why_not"]
}

with open(f"{CONFIG['xai_output_dir']}/explain_schema.json", "w") as f:
    json.dump(explain_schema, f, indent=2)

# Save agreement table
agreement_df.to_csv(
    f"{CONFIG['xai_output_dir']}/comparison/method_agreement.csv", index=False
)

print("Phase 2 outputs saved")
print()
for root, dirs, files in os.walk("outputs/xai"):
    level   = root.replace("outputs/xai", "").count(os.sep)
    indent  = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        print(f"  {indent}{file}")

## 25. Phase 2 Summary

Phase 2 is complete. The explainability engine is built and the interface contract
with Phase 3 is defined.

### What was built

| Component | Method | What it produces |
|-----------|--------|-----------------|
| Spatial explanation | Grad-CAM++ | Heatmap localising disease regions on the leaf |
| Feature attribution | SHAP GradientExplainer | Per-pixel attribution relative to background |
| Decision boundary | LIME | Superpixel mask of locally influential regions |
| Natural language reasoning | Rule-based templates | why / why_not / confidence / counterfactual |
| Counterfactual retrieval | Cosine embedding search | Nearest real image from a different class |
| Method agreement | Spearman correlation | Quantitative Grad-CAM vs SHAP agreement per image |

### What Phase 3 receives

```python
explain(image_path) -> {
    "prediction": {
        "disease"          : "Tomato___Early_blight",
        "plant"            : "Tomato",
        "disease_type"     : "Early blight",
        "confidence"       : 0.876,
        "severity"         : "high",
        "top3_predictions" : [...],
        "feature_embedding": array(shape=(1280,)),
        "is_healthy"       : False
    },
    "gradcam_heatmap"  : array(shape=(224, 224)),
    "gradcam_overlay"  : array(shape=(224, 224, 3)),
    "shap_display"     : array(shape=(224, 224)),
    "lime_overlay"     : array(shape=(224, 224, 3)),
    "explanation": {
        "why"            : "The model predicted Early blight ...",
        "why_not"        : "Late blight was considered but rejected ...",
        "confidence"     : "Confidence is high (87.6%) ...",
        "counterfactual" : "If the lesion regions were absent ..."
    }
}
```

### What Phase 3 will add

Phase 3 builds the treatment planning engine that reads this output and produces a
sequenced, prioritised action plan. The `prediction.disease` and `prediction.severity`
fields drive the rule selection. The `explanation.why` text is embedded directly in
the plan output as rationale, so a farmer reading the plan also understands the
reasoning behind each step.


*Commit this notebook as `notebooks/phase2_explainability_engine.ipynb`. Upload the
`outputs/xai/` directory to Kaggle as `explainplan-phase2-outputs` and attach it to
the Phase 3 notebook alongside the Phase 1 outputs dataset.*